In [4]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import ipywidgets as widgets
from IPython.display import display

ROOT_DIR = "../data/dataset/Dataset420/voi"
IMAGES_DIR = Path(ROOT_DIR) / "images"
MASKS_DIR = Path(ROOT_DIR) / "mask"
SEGS_DIR = Path(ROOT_DIR) / "segmentation"
DECISIONS_FILE = Path(ROOT_DIR) / "voi_decisions.json"

print(f"Root: {ROOT_DIR}")

Root: ../data/dataset/Dataset420/voi


In [5]:
def find_numpy_files(directory: Path) -> List[str]:
    if not directory.exists():
        return []
    return sorted([str(f.relative_to(directory)) for f in directory.rglob("*.npy")])

def load_decisions() -> Dict[str, str]:
    if not DECISIONS_FILE.exists():
        return {}
    with open(DECISIONS_FILE, 'r') as f:
        data = json.load(f)
    return {item['filename']: item['decision'] for item in data}

def save_decisions(decisions: Dict[str, str]):
    output = [{'filename': fn, 'decision': dec} for fn, dec in decisions.items()]
    with open(DECISIONS_FILE, 'w') as f:
        json.dump(output, f, indent=2)

def normalize_slice(slice_data: np.ndarray) -> np.ndarray:
    if slice_data.size == 0:
        return slice_data
    vmin, vmax = np.percentile(slice_data, [1, 99])
    if vmax > vmin:
        normalized = np.clip(slice_data, vmin, vmax)
        normalized = (normalized - vmin) / (vmax - vmin)
    else:
        normalized = np.zeros_like(slice_data)
    return normalized

print("Functions loaded")

Functions loaded


In [6]:
class VOIReviewState:
    def __init__(self):
        self.filenames = find_numpy_files(IMAGES_DIR)
        self.num_cases = len(self.filenames)
        self.case_idx = 0
        self.slice_idx = 0
        self.current_image = None
        self.current_mask = None
        self.current_seg = None
        self.current_filename = None
        self.decisions = load_decisions()
    
    def get_current_filename(self) -> Optional[str]:
        if 0 <= self.case_idx < self.num_cases:
            return self.filenames[self.case_idx]
        return None
    
    def get_decision(self) -> str:
        fn = self.get_current_filename()
        return self.decisions.get(fn, "UNDECIDED") if fn else "UNDECIDED"
    
    def load_current(self) -> bool:
        fn = self.get_current_filename()
        if not fn:
            return False
        
        if fn == self.current_filename and self.current_image is not None:
            return True
        
        try:
            self.current_image = np.load(IMAGES_DIR / fn)
            self.current_mask = np.load(MASKS_DIR / fn)
            self.current_seg = np.load(SEGS_DIR / fn)
            self.current_filename = fn
            self.slice_idx = self.current_image.shape[2] // 2
            return True
        except Exception as e:
            print(f"Error loading {fn}: {e}")
            return False
    
    def get_num_slices(self) -> int:
        return self.current_image.shape[2] if self.current_image is not None else 1
    
    def get_slices(self) -> Optional[Tuple]:
        if self.current_image is None:
            return None
        
        z_dim = self.current_image.shape[2]
        y_dim = self.current_image.shape[1]
        y_center = y_dim // 2
        
        slice_idx_z = max(0, min(self.slice_idx, z_dim - 1))
        
        axial_img = self.current_image[..., slice_idx_z]
        axial_mask = self.current_mask[..., slice_idx_z]
        axial_seg = self.current_seg[..., slice_idx_z]
        
        coronal_img = self.current_image[:, y_center, :]
        coronal_mask = self.current_mask[:, y_center, :]
        coronal_seg = self.current_seg[:, y_center, :]
        
        return (axial_img, axial_mask, axial_seg, coronal_img, coronal_mask, coronal_seg, slice_idx_z, z_dim)
    
    def set_decision(self, decision: str, advance: bool = True):
        fn = self.get_current_filename()
        if not fn:
            return
        self.decisions[fn] = decision
        save_decisions(self.decisions)
        
        if advance and self.case_idx < self.num_cases - 1:
            self.case_idx += 1
            self.load_current()

state = VOIReviewState()
print(f"Loaded {state.num_cases} files, {len(state.decisions)} decisions")

Loaded 164 files, 0 decisions


In [7]:
output = widgets.Output()
slice_slider = widgets.IntSlider(value=0, min=0, max=1, step=1, description='Slice:', continuous_update=False, layout=widgets.Layout(width='600px'))
btn_prev = widgets.Button(description='◀ Previous', button_style='info')
btn_next = widgets.Button(description='Next ▶', button_style='info')
btn_keep = widgets.Button(description='✓ Keep', button_style='success')
btn_delete = widgets.Button(description='✗ Delete', button_style='danger')

def render():
    output.clear_output(wait=True)
    with output:
        if state.current_image is None:
            print(f"No data loaded")
            return
        
        slices = state.get_slices()
        if not slices:
            return
        
        axial_img, axial_mask, axial_seg, coronal_img, coronal_mask, coronal_seg, slice_idx_z, z_dim = slices
        
        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
        axes[0, 0].imshow(normalize_slice(axial_img).T, cmap='gray', origin='lower', aspect='auto')
        axes[0, 0].set_title(f"Axial Image - Slice {slice_idx_z + 1}/{z_dim}", fontsize=12)
        axes[0, 0].axis('off')
        
        axes[0, 1].imshow(axial_mask.T, cmap='jet', origin='lower', aspect='auto')
        axes[0, 1].set_title(f"Axial Mask - Slice {slice_idx_z + 1}/{z_dim}", fontsize=12)
        axes[0, 1].axis('off')
        
        axes[0, 2].imshow(axial_seg.T, cmap='gray', origin='lower', aspect='auto')
        axes[0, 2].set_title(f"Axial Segmentation - Slice {slice_idx_z + 1}/{z_dim}", fontsize=12)
        axes[0, 2].axis('off')
        
        axes[1, 0].imshow(normalize_slice(coronal_img).T, cmap='gray', origin='lower', aspect='auto')
        axes[1, 0].set_title("Coronal Image (Center Y)", fontsize=12)
        axes[1, 0].axis('off')
        h, w = coronal_img.T.shape
        line_y = (slice_idx_z / max(1, z_dim - 1)) * h
        axes[1, 0].axhline(y=line_y, color='yellow', linewidth=2, alpha=0.8)
        
        axes[1, 1].imshow(coronal_mask.T, cmap='jet', origin='lower', aspect='auto')
        axes[1, 1].set_title("Coronal Mask (Center Y)", fontsize=12)
        axes[1, 1].axis('off')
        axes[1, 1].axhline(y=line_y, color='yellow', linewidth=2, alpha=0.8)
        
        axes[1, 2].imshow(coronal_seg.T, cmap='gray', origin='lower', aspect='auto')
        axes[1, 2].set_title("Coronal Segmentation (Center Y)", fontsize=12)
        axes[1, 2].axis('off')
        axes[1, 2].axhline(y=line_y, color='yellow', linewidth=2, alpha=0.8)
        
        plt.tight_layout()
        plt.show()
        plt.close(fig)
        
        fn = state.get_current_filename()
        print(f"\n📁 File: {fn}")
        print(f"📊 Shape: {state.current_image.shape}")
        print(f"📋 Decision: {state.get_decision().upper()}")
        print(f"📍 Case {state.case_idx + 1}/{state.num_cases} | Decisions: {len(state.decisions)}")

def update_slider():
    slice_slider.max = max(0, state.get_num_slices() - 1)
    slice_slider.value = state.slice_idx

def on_slice_change(change):
    state.slice_idx = change['new']
    render()

def on_prev(_):
    if state.case_idx > 0:
        state.case_idx -= 1
        state.load_current()
        update_slider()
        render()

def on_next(_):
    if state.case_idx < state.num_cases - 1:
        state.case_idx += 1
        state.load_current()
        update_slider()
        render()

def on_keep(_):
    state.set_decision("keep", advance=True)
    update_slider()
    render()

def on_delete(_):
    state.set_decision("delete", advance=True)
    update_slider()
    render()

slice_slider.unobserve_all()
slice_slider.observe(on_slice_change, names='value')
btn_prev.on_click(on_prev)
btn_next.on_click(on_next)
btn_keep.on_click(on_keep)
btn_delete.on_click(on_delete)

print("UI ready")

UI ready


In [8]:
ui = widgets.VBox([
    widgets.HBox([btn_prev, btn_next, btn_keep, btn_delete], layout=widgets.Layout(justify_content='center', margin='10px')),
    widgets.HBox([slice_slider], layout=widgets.Layout(justify_content='center', margin='10px')),
    output
])

if state.num_cases > 0:
    state.load_current()
    update_slider()
    render()

display(ui)

In [10]:
def execute_deletions():
    deleted = [fn for fn, dec in state.decisions.items() if dec == "delete"]
    if not deleted:
        print("No files marked for deletion")
        return
    
    for fn in deleted:
        for folder in [IMAGES_DIR, MASKS_DIR, SEGS_DIR]:
            filepath = folder / fn
            if filepath.exists():
                filepath.unlink()
                print(f"Deleted: {filepath}")
    
    print(f"\nTotal files deleted: {len(deleted)}")

execute_deletions()


Total files deleted: 39


In [11]:
def check_and_delete_empty_folders():
    """Check for empty folders in IMAGES_DIR, MASKS_DIR, and SEGS_DIR and delete them"""
    folders_to_check = [IMAGES_DIR, MASKS_DIR, SEGS_DIR]
    
    for folder in folders_to_check:
        if not folder.exists():
            print(f"Folder does not exist: {folder}")
            continue
        
        # Check all subdirectories
        for subfolder in folder.rglob("*"):
            if subfolder.is_dir():
                # Check if directory is empty
                if not any(subfolder.iterdir()):
                    try:
                        subfolder.rmdir()
                        print(f"Deleted empty folder: {subfolder}")
                    except Exception as e:
                        print(f"Error deleting {subfolder}: {e}")
    
    print("\nEmpty folder cleanup complete")

check_and_delete_empty_folders()

Deleted empty folder: ../data/dataset/Dataset420/voi/images/NG/case_00035
Deleted empty folder: ../data/dataset/Dataset420/voi/mask/NG/case_00035
Deleted empty folder: ../data/dataset/Dataset420/voi/segmentation/NG/case_00035

Empty folder cleanup complete
